# Scratch Bi-LSTM

In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from collections import Counter
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A', 'B', 'C', 'D', 'E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'

## 1. Data Loading & Cleaning

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f"Train: {train_df.shape}, Test: {test_df.shape}")
train_df.head()

In [ ]:
def clean_text(t):
    if pd.isna(t):
        return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

## 2. Leak-Free Train/Val Split (UnionFind)

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)
train_df['option_set'] = train_df.apply(option_set_key, axis=1)

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d:
            uf.union(i, d[k])
        else:
            d[k] = i

train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))

train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l: i for i, l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)

y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero overlap in option-sets)")

## 3. Custom Vocabulary & Tokenizer

In [ ]:
def tokenize(text):
    """Simple whitespace tokenizer on cleaned text."""
    return clean_text(text).split()


word_counts = Counter()
for _, row in train_split.iterrows():
    word_counts.update(tokenize(row['prompt']))
    for l in LABELS:
        word_counts.update(tokenize(row[l]))


MAX_VOCAB = 15000
PAD_IDX = 0
UNK_IDX = 1

vocab = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for word, _ in word_counts.most_common(MAX_VOCAB):
    vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

def encode(text, max_len=64):
    """Convert text to token indices, truncate to max_len."""
    tokens = tokenize(text)[:max_len]
    return [vocab.get(w, UNK_IDX) for w in tokens]

## 4. PyTorch Dataset for Multiple Choice

In [ ]:
MAX_PROMPT_LEN = 128
MAX_OPTION_LEN = 64

class MCQDataset(Dataset):
    def __init__(self, df, labels=None):
        self.prompts = []
        self.options = []  
        self.labels = labels

        for _, row in df.iterrows():
            self.prompts.append(encode(row['prompt'], MAX_PROMPT_LEN))
            opts = [encode(row[l], MAX_OPTION_LEN) for l in LABELS]
            self.options.append(opts)

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = torch.tensor(self.prompts[idx], dtype=torch.long)
        options = [torch.tensor(o, dtype=torch.long) for o in self.options[idx]]
        if self.labels is not None:
            return prompt, options, self.labels[idx]
        return prompt, options


def collate_fn(batch):
    """Custom collation: pad prompts and each option separately."""
    has_labels = len(batch[0]) == 3

    prompts = [b[0] for b in batch]
    all_options = [b[1] for b in batch]

    
    prompt_lens = torch.tensor([len(p) for p in prompts])
    prompts_padded = pad_sequence(prompts, batch_first=True, padding_value=PAD_IDX)

    
    options_padded = []
    option_lens = []
    for opt_idx in range(5):
        opts = [b[opt_idx] for b in all_options]
        lens = torch.tensor([max(len(o), 1) for o in opts])
    
        opts = [o if len(o) > 0 else torch.tensor([UNK_IDX]) for o in opts]
        padded = pad_sequence(opts, batch_first=True, padding_value=PAD_IDX)
        options_padded.append(padded)
        option_lens.append(lens)

    if has_labels:
        labels = torch.tensor([b[2] for b in batch], dtype=torch.long)
        return prompts_padded, prompt_lens, options_padded, option_lens, labels

    return prompts_padded, prompt_lens, options_padded, option_lens